In [1]:
"""Leakage check: rebuild each *_before feature from tournament results and compare with the CSV.
Run in the same folder as regression_2_2_dataset.csv."""
import pandas as pd, numpy as np
df = pd.read_csv("regression_2_2_dataset.csv", parse_dates=["match_date"])
# 1) FIFA points: one value per team? (a single pre-tournament snapshot) and mirrored correctly?
print("Teams whose team_fifa_points changes across matches:", int((df.groupby("team_id").team_fifa_points.nunique() > 1).sum()))
mm = df.merge(df, left_on=["match_id", "opponent_id"], right_on=["match_id", "team_id"], suffixes=("", "_o"))
print("opponent_fifa_points mirrors the opponent's team_fifa_points:", np.allclose(mm.opponent_fifa_points, mm.team_fifa_points_o))
# 2) Rebuild goals conceded from the opponent's row
o = df[["match_id", "team_id", "goals_scored"]].merge(df[["match_id", "team_id", "goals_scored"]], on="match_id", suffixes=("", "_o"))
o = o[o.team_id != o.team_id_o].rename(columns={"goals_scored_o": "conceded"})
d = df.merge(o[["match_id", "team_id", "conceded"]], on=["match_id", "team_id"]).sort_values(["team_id", "match_date"]).reset_index(drop=True)
g = d.groupby("team_id"); d["k"] = g.cumcount()
d["win"] = (d.goals_scored > d.conceded) * 1.0; d["gd"] = d.goals_scored - d.conceded
d["cs"] = (d.conceded == 0) * 1.0; d["sc"] = (d.goals_scored > 0) * 1.0
prior = lambda c: g[c].transform(lambda s: s.shift(1).expanding().mean())   # PRIOR matches only
for a, c in [("r_gs", "goals_scored"), ("r_win", "win"), ("r_gd", "gd"), ("r_conc", "conceded"), ("r_cs", "cs"), ("r_sc", "sc")]:
    d[a] = prior(c)
late = d[d.k >= 1]
for f, r in [("avg_goals_scored_before", "r_gs"), ("win_rate_before", "r_win"), ("avg_goal_difference_before", "r_gd")]:
    print(f, "max |diff| vs prior-only rebuild (rows after a team's 1st match):", round((late[f] - late[r]).abs().max(), 6))
opp = d[["match_id", "team_id", "k", "r_conc", "r_cs", "r_sc"]].rename(columns={"team_id": "opponent_id", "k": "k_o"})
x = d.merge(opp, on=["match_id", "opponent_id"], suffixes=("", "_opp")); x = x[x.k_o >= 1]
for f, r in [("opponent_avg_goals_conceded_before", "r_conc_opp"), ("opponent_clean_sheet_rate_before", "r_cs_opp"), ("opponent_scoring_match_rate_before", "r_sc_opp")]:
    print(f, "max |diff| vs rebuild (opponent past 1st match):", round((x[f] - x[r]).abs().max(), 6))
print("A team's FIRST match uses a pre-tournament window that cannot be rebuilt from this file ->",
      "check the data-collection code: all those matches must be dated before", df.match_date.min())

/Users/anishbhattarai/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/anishbhattarai/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


Teams whose team_fifa_points changes across matches: 0
opponent_fifa_points mirrors the opponent's team_fifa_points: True
avg_goals_scored_before max |diff| vs prior-only rebuild (rows after a team's 1st match): 0.0
win_rate_before max |diff| vs prior-only rebuild (rows after a team's 1st match): 0.0
avg_goal_difference_before max |diff| vs prior-only rebuild (rows after a team's 1st match): 0.0
opponent_avg_goals_conceded_before max |diff| vs rebuild (opponent past 1st match): 0.0
opponent_clean_sheet_rate_before max |diff| vs rebuild (opponent past 1st match): 0.0
opponent_scoring_match_rate_before max |diff| vs rebuild (opponent past 1st match): 0.0
A team's FIRST match uses a pre-tournament window that cannot be rebuilt from this file -> check the data-collection code: all those matches must be dated before 2026-06-11 19:00:00+00:00
